# Micrograd Cold Rebuild
### In this notebook, I will attempt to rebuild the core components of micrograd from scratch.
### The goal is to write the Value class, implement the backward pass for autodifferentiation, and construct a simple MLP, all without referring back to the original code. This will serve as a revision and a test of my understanding of the fundamentals.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
import random

In [2]:
class Value:
    def __init__(self, data, _children=set(), _op=''):
        self.data = data
        self.grad = 0.0
        self._prev = _children
        self._op = _op
        self._backward = lambda: None
    
    def __repr__(self):
        return f'Value(data={self.data})'
    
    def __add__(self, other):
        other = Value(other) if not isinstance(other, Value) else other
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out
    
    def __radd__(self, other):
        return self + other
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def __rmul__(self, other):
        return self * other
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    
    def __rsub__(self, other):
        return -self + other
    
    def exp(self):
        out = Value(math.exp(self.data), (self, ), 'exp')
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out
    
    def __pow__(self, x):
        assert isinstance(x, (int, float)), 'only int and float exponent supported for now'
        out = Value(self.data ** x, (self, ), 'pow')
        def _backward():
            self.grad += x * (self.data ** (x - 1)) * out.grad
        out._backward = _backward
        return out

    def __truediv__(self, other):
        return self * (other ** -1)
    
    def __rtruediv__(self, other):
        return (self ** -1) * other
    
    def tanh(self):
        t = math.exp(2*self.data)
        out = Value((t - 1) / (t + 1), (self, ), 'tanh')
        def _backward():
            self.grad += (1 - out.data**2) * out.grad
        out._backward = _backward
        return out
    
    def backward(self, setgrad=False):
        if setgrad: self.grad = 1.0
        self._backward()
        for c in self._prev:
            c.backward()

In [3]:
# --- Value implementation tests (read-only: does not modify your Value class) ---

def approx(a, b, tol=1e-6):
    return abs(a - b) < tol

def check(name, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    line = f"[{status}] {name}"
    if detail:
        line += f" — {detail}"
    print(line)
    return condition

def run_test(name, fn):
    try:
        ok = fn()
        if ok is False:
            print(f"[FAIL] {name}")
        elif ok is not False:
            print(f"[PASS] {name}")
    except Exception as e:
        print(f"[FAIL] {name} — {type(e).__name__}: {e}")

print("=" * 60)
print("FORWARD PASS")
print("=" * 60)

a, b = Value(2.0), Value(3.0)
check("add", approx((a + b).data, 5.0), f"expected 5.0, got {(a + b).data}")
check("mul", approx((a * b).data, 6.0), f"expected 6.0, got {(a * b).data}")
check("sub", approx((a - b).data, -1.0), f"expected -1.0, got {(a - b).data}")
check("neg", approx((-a).data, -2.0), f"expected -2.0, got {(-a).data}")
check("radd (scalar + Value)", approx((5 + a).data, 7.0), f"expected 7.0, got {(5 + a).data}")
check("rmul (scalar * Value)", approx((4 * b).data, 12.0), f"expected 12.0, got {(4 * b).data}")
check("pow", approx((a ** 2).data, 4.0), f"expected 4.0, got {(a ** 2).data}")
check("exp", approx(Value(0.0).exp().data, 1.0), f"expected 1.0, got {Value(0.0).exp().data}")
check("div", approx((b / a).data, 1.5), f"expected 1.5, got {(b / a).data}")
check("rdiv (scalar / Value)", approx((6 / a).data, 3.0), f"expected 3.0, got {(6 / a).data}")

print("\n" + "=" * 60)
print("VALUE + SCALAR (mixed operands)")
print("=" * 60)

x = Value(2.0)
check("Value + int", approx((x + 3).data, 5.0), f"expected 5.0, got {(x + 3).data}")
check("Value + float", approx((x + 1.5).data, 3.5), f"expected 3.5, got {(x + 1.5).data}")
check("int + Value", approx((3 + x).data, 5.0), f"expected 5.0, got {(3 + x).data}")
check("float + Value", approx((1.5 + x).data, 3.5), f"expected 3.5, got {(1.5 + x).data}")
check("Value * int", approx((x * 4).data, 8.0), f"expected 8.0, got {(x * 4).data}")
check("Value * float", approx((x * 2.5).data, 5.0), f"expected 5.0, got {(x * 2.5).data}")
check("int * Value", approx((4 * x).data, 8.0), f"expected 8.0, got {(4 * x).data}")
check("float * Value", approx((2.5 * x).data, 5.0), f"expected 5.0, got {(2.5 * x).data}")
check("Value - int", approx((x - 1).data, 1.0), f"expected 1.0, got {(x - 1).data}")
check("Value - float", approx((x - 0.5).data, 1.5), f"expected 1.5, got {(x - 0.5).data}")
check("int - Value", approx((5 - x).data, 3.0), f"expected 3.0, got {(5 - x).data}")
check("float - Value", approx((5.5 - x).data, 3.5), f"expected 3.5, got {(5.5 - x).data}")
check("Value / int", approx((Value(6.0) / 2).data, 3.0), f"expected 3.0, got {(Value(6.0) / 2).data}")
check("Value / float", approx((Value(6.0) / 2.0).data, 3.0), f"expected 3.0, got {(Value(6.0) / 2.0).data}")
check("int / Value", approx((8 / x).data, 4.0), f"expected 4.0, got {(8 / x).data}")
check("float / Value", approx((5.0 / x).data, 2.5), f"expected 2.5, got {(5.0 / x).data}")

print("\n" + "=" * 60)
print("BACKWARD PASS")
print("=" * 60)

def test_mul_backward():
    x = Value(3.0)
    y = Value(4.0)
    z = x * y
    z.backward(setgrad=True)
    return check("d(x*y)/dx and d(x*y)/dy", approx(x.grad, 4.0) and approx(y.grad, 3.0),
                 f"x.grad={x.grad}, y.grad={y.grad}")

def test_add_backward():
    x = Value(2.0)
    y = Value(5.0)
    z = x + y
    z.backward(setgrad=True)
    return check("d(x+y)/dx and d(x+y)/dy", approx(x.grad, 1.0) and approx(y.grad, 1.0),
                 f"x.grad={x.grad}, y.grad={y.grad}")

def test_chain_rule():
    # L = (2*x + 3)^2  at x=4  => L=121, dL/dx = 2*(2x+3)*2 = 4*(2*4+3) = 44
    x = Value(4.0)
    L = (2 * x + 3) ** 2
    L.backward(setgrad=True)
    return check("chain rule on (2x+3)^2", approx(x.grad, 44.0), f"x.grad={x.grad}")

def test_karpathy_example():
    # L = (a*b + c) * f
    a = Value(2.0)
    b = Value(-3.0)
    c = Value(10.0)
    f = Value(-2.0)
    L = (a * b + c) * f
    L.backward(setgrad=True)
    ok = (
        approx(a.grad, 6.0) and
        approx(b.grad, -4.0) and
        approx(c.grad, -2.0) and
        approx(f.grad, 4.0)
    )
    return check(
        "Karpathy example: L = (a*b + c) * f",
        ok,
        f"a.grad={a.grad}, b.grad={b.grad}, c.grad={c.grad}, f.grad={f.grad}"
    )

def test_neuron():
    # x1*w1 + x2*w2 + b, then tanh — Karpathy lecture 1 reference values
    x1, x2 = Value(2.0), Value(0.0)
    w1, w2 = Value(-3.0), Value(1.0)
    b = Value(6.881373587019543)
    n = x1 * w1 + x2 * w2 + b
    o = n.tanh()
    o.backward(setgrad=True)
    ok = (
        approx(o.data, 0.7071, tol=1e-4) and
        approx(x1.grad, -1.5, tol=1e-4) and
        approx(x2.grad, 0.5, tol=1e-4) and
        approx(w1.grad, 1.0, tol=1e-4) and
        approx(w2.grad, 0.0, tol=1e-4) and
        approx(b.grad, 0.5, tol=1e-4)
    )
    return check(
        "neuron: tanh(x1*w1 + x2*w2 + b)",
        ok,
        f"o={o.data:.4f}, x1.grad={x1.grad}, x2.grad={x2.grad}, "
        f"w1.grad={w1.grad}, w2.grad={w2.grad}, b.grad={b.grad}"
    )

def test_reused_node():
    # b = a + a  => db/da should accumulate to 2
    a = Value(3.0)
    b = a + a
    b.backward(setgrad=True)
    return check("gradient accumulation (a used twice)", approx(a.grad, 2.0), f"a.grad={a.grad}")

def test_scalar_add_backward():
    x = Value(2.0)
    L = x + 5          # Value + int
    L.backward(setgrad=True)
    return check("d(x+5)/dx", approx(x.grad, 1.0), f"x.grad={x.grad}")

def test_scalar_radd_backward():
    x = Value(2.0)
    L = 3 + x          # int + Value
    L.backward(setgrad=True)
    return check("d(3+x)/dx", approx(x.grad, 1.0), f"x.grad={x.grad}")

def test_scalar_mul_backward():
    x = Value(2.0)
    L = x * 4          # Value * int
    L.backward(setgrad=True)
    return check("d(x*4)/dx", approx(x.grad, 4.0), f"x.grad={x.grad}")

def test_scalar_rmul_backward():
    x = Value(2.0)
    L = 3.0 * x        # float * Value
    L.backward(setgrad=True)
    return check("d(3*x)/dx", approx(x.grad, 3.0), f"x.grad={x.grad}")

def test_scalar_sub_backward():
    x = Value(2.0)
    L = x - 1          # Value - int
    L.backward(setgrad=True)
    return check("d(x-1)/dx", approx(x.grad, 1.0), f"x.grad={x.grad}")

def test_scalar_rsub_backward():
    x = Value(2.0)
    L = 10 - x         # int - Value
    L.backward(setgrad=True)
    return check("d(10-x)/dx", approx(x.grad, -1.0), f"x.grad={x.grad}")

def test_scalar_div_backward():
    x = Value(4.0)
    L = x / 2          # Value / int
    L.backward(setgrad=True)
    return check("d(x/2)/dx", approx(x.grad, 0.5), f"x.grad={x.grad}")

def test_scalar_rdiv_backward():
    x = Value(2.0)
    L = 6 / x          # int / Value
    L.backward(setgrad=True)
    return check("d(6/x)/dx", approx(x.grad, -1.5), f"x.grad={x.grad}")

for fn in [
    test_add_backward,
    test_mul_backward,
    test_chain_rule,
    test_karpathy_example,
    test_reused_node,
    test_scalar_add_backward,
    test_scalar_radd_backward,
    test_scalar_mul_backward,
    test_scalar_rmul_backward,
    test_scalar_sub_backward,
    test_scalar_rsub_backward,
    test_scalar_div_backward,
    test_scalar_rdiv_backward,
    test_neuron,
]:
    run_test(fn.__name__, fn)

print("\nDone. Any [FAIL] lines point to ops or backward logic that need work.")

FORWARD PASS
[PASS] add — expected 5.0, got 5.0
[PASS] mul — expected 6.0, got 6.0
[PASS] sub — expected -1.0, got -1.0
[PASS] neg — expected -2.0, got -2.0
[PASS] radd (scalar + Value) — expected 7.0, got 7.0
[PASS] rmul (scalar * Value) — expected 12.0, got 12.0
[PASS] pow — expected 4.0, got 4.0
[PASS] exp — expected 1.0, got 1.0
[PASS] div — expected 1.5, got 1.5
[PASS] rdiv (scalar / Value) — expected 3.0, got 3.0

VALUE + SCALAR (mixed operands)
[PASS] Value + int — expected 5.0, got 5.0
[PASS] Value + float — expected 3.5, got 3.5
[PASS] int + Value — expected 5.0, got 5.0
[PASS] float + Value — expected 3.5, got 3.5
[PASS] Value * int — expected 8.0, got 8.0
[PASS] Value * float — expected 5.0, got 5.0
[PASS] int * Value — expected 8.0, got 8.0
[PASS] float * Value — expected 5.0, got 5.0
[PASS] Value - int — expected 1.0, got 1.0
[PASS] Value - float — expected 1.5, got 1.5
[PASS] int - Value — expected 3.0, got 3.0
[PASS] float - Value — expected 3.5, got 3.5
[PASS] Value / i

In [4]:
class Neuron:
    def __init__(self, num_inputs):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(num_inputs)]
        self.b = Value(random.uniform(-1, 1))
    
    def __call__(self, x):
        out = sum(wi * xi for wi, xi in zip(self.w, x)) + self.b
        act = out.tanh()
        return act
    
    def parameters(self):
        return [x for x in self.w] + [self.b]

In [5]:
# --- Neuron implementation tests (read-only: does not modify your Neuron class) ---

def approx(a, b, tol=1e-6):
    return abs(a - b) < tol

def check(name, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    line = f"[{status}] {name}"
    if detail:
        line += f" — {detail}"
    print(line)
    return condition

def run_test(name, fn):
    try:
        ok = fn()
        if ok is False:
            print(f"[FAIL] {name}")
        elif ok is not False:
            print(f"[PASS] {name}")
    except Exception as e:
        print(f"[FAIL] {name} — {type(e).__name__}: {e}")

def make_neuron(w_values, b_value):
    n = Neuron(len(w_values))
    n.w = [Value(w) for w in w_values]
    n.b = Value(b_value)
    return n

print("=" * 60)
print("NEURON — STRUCTURE")
print("=" * 60)

def test_init_num_weights():
    n = Neuron(3)
    return check("Neuron(3) creates 3 weights", len(n.w) == 3, f"len(w)={len(n.w)}")

def test_parameters_count():
    n = Neuron(2)
    return check("parameters() returns weights + bias", len(n.parameters()) == 3,
                 f"got {len(n.parameters())} params")

def test_parameters_are_values():
    n = Neuron(2)
    return check("parameters() are Value objects",
                 all(isinstance(p, Value) for p in n.parameters()))

for fn in [test_init_num_weights, test_parameters_count, test_parameters_are_values]:
    run_test(fn.__name__, fn)

print("\n" + "=" * 60)
print("NEURON — FORWARD PASS")
print("=" * 60)

def test_forward_two_inputs_with_tanh():
    n = make_neuron([-3.0, 1.0], 6.881373587019543)
    x = [Value(2.0), Value(0.0)]
    out = n(x)
    return check("forward: tanh(w·x + b)", approx(out.data, 0.7071, tol=1e-4),
                 f"expected 0.7071, got {out.data:.4f}")

def test_forward_single_input_with_tanh():
    n = make_neuron([2.5], -1.0)
    out = n([Value(4.0)])
    expected = math.tanh(2.5 * 4.0 - 1.0)
    return check("forward: single input with tanh", approx(out.data, expected, tol=1e-6),
                 f"expected {expected:.6f}, got {out.data:.6f}")

def test_forward_scalar_inputs_with_tanh():
    n = make_neuron([2.0, 3.0], 1.0)
    out = n([1.0, 2.0])
    expected = math.tanh(2.0 * 1.0 + 3.0 * 2.0 + 1.0)
    return check("forward: scalar inputs with tanh", approx(out.data, expected, tol=1e-6),
                 f"expected {expected:.6f}, got {out.data:.6f}")

for fn in [
    test_forward_two_inputs_with_tanh,
    test_forward_single_input_with_tanh,
    test_forward_scalar_inputs_with_tanh,
]:
    run_test(fn.__name__, fn)

print("\n" + "=" * 60)
print("NEURON — BACKWARD PASS")
print("=" * 60)

def test_backward_karpathy_neuron():
    n = make_neuron([-3.0, 1.0], 6.881373587019543)
    x = [Value(2.0), Value(0.0)]
    out = n(x)
    out.backward(setgrad=True)
    ok = (
        approx(out.data, 0.7071, tol=1e-4) and
        approx(x[0].grad, -1.5, tol=1e-4) and
        approx(x[1].grad, 0.5, tol=1e-4) and
        approx(n.w[0].grad, 1.0, tol=1e-4) and
        approx(n.w[1].grad, 0.0, tol=1e-4) and
        approx(n.b.grad, 0.5, tol=1e-4)
    )
    return check(
        "backward: tanh neuron reference case",
        ok,
        f"out={out.data:.4f}, x1.grad={x[0].grad}, x2.grad={x[1].grad}, "
        f"w1.grad={n.w[0].grad}, w2.grad={n.w[1].grad}, b.grad={n.b.grad}"
    )

def test_backward_single_input_with_tanh():
    n = make_neuron([2.5], -1.0)
    x = [Value(4.0)]
    out = n(x)
    out.backward(setgrad=True)
    preact = 2.5 * 4.0 - 1.0
    local_grad = 1 - math.tanh(preact) ** 2
    ok = (
        approx(x[0].grad, 2.5 * local_grad, tol=1e-6) and
        approx(n.w[0].grad, 4.0 * local_grad, tol=1e-6) and
        approx(n.b.grad, 1.0 * local_grad, tol=1e-6)
    )
    return check(
        "backward: single input with tanh",
        ok,
        f"x.grad={x[0].grad}, w.grad={n.w[0].grad}, b.grad={n.b.grad}"
    )

def test_backward_scalar_inputs_with_tanh():
    n = make_neuron([2.0, 3.0], 1.0)
    out = n([1.0, 2.0])
    out.backward(setgrad=True)
    preact = 2.0 * 1.0 + 3.0 * 2.0 + 1.0
    local_grad = 1 - math.tanh(preact) ** 2
    ok = (
        approx(n.w[0].grad, 1.0 * local_grad, tol=1e-6) and
        approx(n.w[1].grad, 2.0 * local_grad, tol=1e-6) and
        approx(n.b.grad, 1.0 * local_grad, tol=1e-6)
    )
    return check(
        "backward: scalar inputs with tanh",
        ok,
        f"w1.grad={n.w[0].grad}, w2.grad={n.w[1].grad}, b.grad={n.b.grad}"
    )

for fn in [
    test_backward_karpathy_neuron,
    test_backward_single_input_with_tanh,
    test_backward_scalar_inputs_with_tanh,
]:
    run_test(fn.__name__, fn)

print("\nDone. Any [FAIL] lines point to Neuron logic that needs work.")

NEURON — STRUCTURE
[PASS] Neuron(3) creates 3 weights — len(w)=3
[PASS] test_init_num_weights
[PASS] parameters() returns weights + bias — got 3 params
[PASS] test_parameters_count
[PASS] parameters() are Value objects
[PASS] test_parameters_are_values

NEURON — FORWARD PASS
[PASS] forward: tanh(w·x + b) — expected 0.7071, got 0.7071
[PASS] test_forward_two_inputs_with_tanh
[PASS] forward: single input with tanh — expected 1.000000, got 1.000000
[PASS] test_forward_single_input_with_tanh
[PASS] forward: scalar inputs with tanh — expected 1.000000, got 1.000000
[PASS] test_forward_scalar_inputs_with_tanh

NEURON — BACKWARD PASS
[PASS] backward: tanh neuron reference case — out=0.7071, x1.grad=-1.4999999999999996, x2.grad=0.4999999999999999, w1.grad=0.9999999999999998, w2.grad=0.0, b.grad=0.4999999999999999
[PASS] test_backward_karpathy_neuron
[PASS] backward: single input with tanh — x.grad=1.5229979277719963e-07, w.grad=2.436796684435194e-07, b.grad=6.091991711087985e-08
[PASS] test_ba

In [6]:
class Layer:
    def __init__(self, num_inputs, num_outputs):
        self.neurons = [Neuron(num_inputs) for _ in range(num_outputs)]
    
    def __call__(self, x):
        acts = [n(x) for n in self.neurons]
        return acts
    
    def parameters(self):
        p = []
        for n in self.neurons:
            p.extend(n.parameters())
        return p

In [7]:
# --- Layer implementation tests (read-only: does not modify your Layer class) ---

def approx(a, b, tol=1e-6):
    return abs(a - b) < tol

def check(name, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    line = f"[{status}] {name}"
    if detail:
        line += f" — {detail}"
    print(line)
    return condition

def run_test(name, fn):
    try:
        ok = fn()
        if ok is False:
            print(f"[FAIL] {name}")
        elif ok is not False:
            print(f"[PASS] {name}")
    except Exception as e:
        print(f"[FAIL] {name} — {type(e).__name__}: {e}")

def make_layer(weight_rows, bias_values):
    layer = Layer(len(weight_rows[0]), len(weight_rows))
    layer.neurons = []
    for w_values, b_value in zip(weight_rows, bias_values):
        n = Neuron(len(w_values))
        n.w = [Value(w) for w in w_values]
        n.b = Value(b_value)
        layer.neurons.append(n)
    return layer

print("=" * 60)
print("LAYER — STRUCTURE")
print("=" * 60)

def test_init_num_neurons():
    layer = Layer(3, 4)
    return check("Layer(3, 4) creates 4 neurons", len(layer.neurons) == 4,
                 f"len(neurons)={len(layer.neurons)}")

def test_all_neurons_are_neuron_objects():
    layer = Layer(2, 3)
    return check("all members are Neuron objects",
                 all(isinstance(n, Neuron) for n in layer.neurons))

for fn in [test_init_num_neurons, test_all_neurons_are_neuron_objects]:
    run_test(fn.__name__, fn)

print("\n" + "=" * 60)
print("LAYER — FORWARD PASS")
print("=" * 60)

def test_forward_returns_list_of_outputs():
    layer = make_layer([[1.0, 2.0], [-3.0, 1.0]], [0.5, 6.881373587019543])
    x = [Value(1.0), Value(2.0)]
    out = layer(x)
    ok = isinstance(out, list) and len(out) == 2 and all(isinstance(v, Value) for v in out)
    return check("forward returns list[Value]", ok,
                 f"type={type(out).__name__}, len={len(out) if isinstance(out, list) else 'n/a'}")

def test_forward_expected_values():
    layer = make_layer([[1.0, 2.0], [-3.0, 1.0]], [0.5, 6.881373587019543])
    x = [Value(1.0), Value(2.0)]
    out = layer(x)
    expected0 = math.tanh(1.0 * 1.0 + 2.0 * 2.0 + 0.5)
    expected1 = math.tanh(-3.0 * 1.0 + 1.0 * 2.0 + 6.881373587019543)
    ok = approx(out[0].data, expected0, tol=1e-6) and approx(out[1].data, expected1, tol=1e-6)
    return check("forward computes each neuron output", ok,
                 f"got [{out[0].data:.6f}, {out[1].data:.6f}]")

def test_forward_scalar_inputs():
    layer = make_layer([[2.0, -1.0], [0.5, 3.0]], [1.0, -2.0])
    out = layer([3.0, 4.0])
    expected0 = math.tanh(2.0 * 3.0 + (-1.0) * 4.0 + 1.0)
    expected1 = math.tanh(0.5 * 3.0 + 3.0 * 4.0 - 2.0)
    ok = approx(out[0].data, expected0, tol=1e-6) and approx(out[1].data, expected1, tol=1e-6)
    return check("forward works with scalar inputs", ok,
                 f"got [{out[0].data:.6f}, {out[1].data:.6f}]")

for fn in [test_forward_returns_list_of_outputs, test_forward_expected_values, test_forward_scalar_inputs]:
    run_test(fn.__name__, fn)

print("\n" + "=" * 60)
print("LAYER — PARAMETERS")
print("=" * 60)

def test_parameters_returns_all_params():
    layer = make_layer([[1.0, 2.0], [-3.0, 1.0]], [0.5, 6.0])
    params = layer.parameters()
    expected_count = 2 * 2 + 2  # 2 neurons, 2 weights each, 1 bias each
    return check("parameters returns all neuron params", len(params) == expected_count,
                 f"expected {expected_count}, got {len(params)}")

def test_parameters_are_values():
    layer = make_layer([[1.0, 2.0], [-3.0, 1.0]], [0.5, 6.0])
    params = layer.parameters()
    return check("parameters are Value objects", all(isinstance(p, Value) for p in params))

for fn in [test_parameters_returns_all_params, test_parameters_are_values]:
    run_test(fn.__name__, fn)

print("\nDone. Any [FAIL] lines point to Layer logic that needs work.")

LAYER — STRUCTURE
[PASS] Layer(3, 4) creates 4 neurons — len(neurons)=4
[PASS] test_init_num_neurons
[PASS] all members are Neuron objects
[PASS] test_all_neurons_are_neuron_objects

LAYER — FORWARD PASS
[PASS] forward returns list[Value] — type=list, len=2
[PASS] test_forward_returns_list_of_outputs
[PASS] forward computes each neuron output — got [0.999967, 0.999984]
[PASS] test_forward_expected_values
[PASS] forward works with scalar inputs — got [0.995055, 1.000000]
[PASS] test_forward_scalar_inputs

LAYER — PARAMETERS
[PASS] parameters returns all neuron params — expected 6, got 6
[PASS] test_parameters_returns_all_params
[PASS] parameters are Value objects
[PASS] test_parameters_are_values

Done. Any [FAIL] lines point to Layer logic that needs work.


In [8]:
class MLP:
    def __init__(self, num_inputs, list_num_outputs):
        sizes = [num_inputs] + list_num_outputs
        self.layers = [Layer(sizes[i], sizes[i + 1]) for i in range(len(sizes) - 1)]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for l in self.layers for p in l.parameters()]


In [9]:
# --- MLP implementation tests (read-only: does not modify your MLP class) ---

def approx(a, b, tol=1e-6):
    return abs(a - b) < tol

def check(name, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    line = f"[{status}] {name}"
    if detail:
        line += f" — {detail}"
    print(line)
    return condition

def run_test(name, fn):
    try:
        ok = fn()
        if ok is False:
            print(f"[FAIL] {name}")
        elif ok is not False:
            print(f"[PASS] {name}")
    except Exception as e:
        print(f"[FAIL] {name} — {type(e).__name__}: {e}")

def make_mlp(layer_specs):
    """
    layer_specs: list of (weight_rows, bias_values) for each layer.
    weight_rows[j] are the weights of neuron j in that layer.
    """
    nin = len(layer_specs[0][0][0])
    nouts = [len(ws) for ws, _ in layer_specs]
    mlp = MLP(nin, nouts)
    mlp.layers = []
    for weight_rows, bias_values in layer_specs:
        layer = Layer(len(weight_rows[0]), len(weight_rows))
        layer.neurons = []
        for w_values, b_value in zip(weight_rows, bias_values):
            n = Neuron(len(w_values))
            n.w = [Value(w) for w in w_values]
            n.b = Value(b_value)
            layer.neurons.append(n)
        mlp.layers.append(layer)
    return mlp

print("=" * 60)
print("MLP — STRUCTURE")
print("=" * 60)

def test_init_num_layers():
    # MLP(3, [4, 4, 1]) should build sizes [3, 4, 4, 1] => 3 layers
    mlp = MLP(3, [4, 4, 1])
    return check("MLP(3, [4, 4, 1]) creates 3 layers", len(mlp.layers) == 3,
                 f"len(layers)={len(mlp.layers)}")

def test_layer_sizes():
    mlp = MLP(2, [3, 1])
    sizes = [(len(layer.neurons[0].w), len(layer.neurons)) for layer in mlp.layers]
    expected = [(2, 3), (3, 1)]
    return check("layer sizes match [2]->[3]->[1]", sizes == expected,
                 f"got {sizes}")

def test_layers_are_layer_objects():
    mlp = MLP(2, [2, 1])
    return check("all members are Layer objects",
                 all(isinstance(l, Layer) for l in mlp.layers))

for fn in [test_init_num_layers, test_layer_sizes, test_layers_are_layer_objects]:
    run_test(fn.__name__, fn)

print("\n" + "=" * 60)
print("MLP — FORWARD PASS")
print("=" * 60)

def test_forward_output_shape():
    # 2 -> 3 -> 1 network; final layer has 1 neuron => list of length 1
    mlp = make_mlp([
        ([[0.5, -0.5], [1.0, 0.0], [0.0, 1.0]], [0.1, -0.2, 0.3]),
        ([[1.0, -1.0, 0.5]], [0.0]),
    ])
    out = mlp([Value(1.0), Value(-1.0)])
    ok = isinstance(out, list) and len(out) == 1 and isinstance(out[0], Value)
    return check("forward returns list[Value] of final layer size", ok,
                 f"type={type(out).__name__}, len={len(out) if isinstance(out, list) else 'n/a'}")

def test_forward_expected_values():
    # Hand-computed 2 -> 2 -> 1 network
    mlp = make_mlp([
        ([[1.0, 0.0], [0.0, 1.0]], [0.0, 0.0]),
        ([[1.0, 1.0]], [0.0]),
    ])
    x = [Value(0.5), Value(-0.5)]
    out = mlp(x)
    h0 = math.tanh(1.0 * 0.5 + 0.0 * (-0.5) + 0.0)
    h1 = math.tanh(0.0 * 0.5 + 1.0 * (-0.5) + 0.0)
    expected = math.tanh(1.0 * h0 + 1.0 * h1 + 0.0)
    return check("forward matches stacked tanh layers", approx(out[0].data, expected, tol=1e-6),
                 f"expected {expected:.6f}, got {out[0].data:.6f}")

def test_forward_scalar_inputs():
    mlp = make_mlp([
        ([[2.0, -1.0]], [0.5]),
        ([[1.5]], [-0.25]),
    ])
    out = mlp([1.0, 2.0])
    h = math.tanh(2.0 * 1.0 + (-1.0) * 2.0 + 0.5)
    expected = math.tanh(1.5 * h - 0.25)
    return check("forward works with scalar inputs", approx(out[0].data, expected, tol=1e-6),
                 f"expected {expected:.6f}, got {out[0].data:.6f}")

def test_forward_multi_output():
    # Final layer with 2 neurons
    mlp = make_mlp([
        ([[1.0], [2.0]], [0.0, 0.0]),
        ([[1.0, 0.0], [0.0, 1.0]], [0.0, 0.0]),
    ])
    out = mlp([Value(0.5)])
    e0 = math.tanh(1.0 * math.tanh(1.0 * 0.5) + 0.0 * math.tanh(2.0 * 0.5))
    e1 = math.tanh(0.0 * math.tanh(1.0 * 0.5) + 1.0 * math.tanh(2.0 * 0.5))
    ok = len(out) == 2 and approx(out[0].data, e0) and approx(out[1].data, e1)
    return check("forward multi-output final layer", ok,
                 f"got [{out[0].data:.6f}, {out[1].data:.6f}]")

for fn in [
    test_forward_output_shape,
    test_forward_expected_values,
    test_forward_scalar_inputs,
    test_forward_multi_output,
]:
    run_test(fn.__name__, fn)

print("\n" + "=" * 60)
print("MLP — PARAMETERS")
print("=" * 60)

def test_parameters_count():
    # 2 -> 3 -> 1 :
    # layer1: 3 neurons * (2 weights + 1 bias) = 9
    # layer2: 1 neuron  * (3 weights + 1 bias) = 4
    # total = 13
    mlp = make_mlp([
        ([[0.1, 0.2], [0.3, 0.4], [0.5, 0.6]], [0.0, 0.0, 0.0]),
        ([[0.7, 0.8, 0.9]], [1.0]),
    ])
    params = mlp.parameters()
    return check("parameters returns all layer params", len(params) == 13,
                 f"expected 13, got {len(params)}")

def test_parameters_are_values():
    mlp = make_mlp([
        ([[1.0, 2.0]], [0.0]),
        ([[3.0]], [0.0]),
    ])
    params = mlp.parameters()
    return check("parameters are Value objects", all(isinstance(p, Value) for p in params))

def test_parameters_match_init_count():
    # Built via constructor: MLP(3, [4, 1])
    # layer1: 4*(3+1)=16, layer2: 1*(4+1)=5, total=21
    # (If __init__ builds the wrong number of layers, this will fail.)
    mlp = MLP(3, [4, 1])
    return check("MLP(3, [4, 1]) has 21 parameters", len(mlp.parameters()) == 21,
                 f"got {len(mlp.parameters())}")

for fn in [test_parameters_count, test_parameters_are_values, test_parameters_match_init_count]:
    run_test(fn.__name__, fn)

print("\n" + "=" * 60)
print("MLP — BACKWARD PASS")
print("=" * 60)

def test_backward_flows_to_inputs_and_params():
    mlp = make_mlp([
        ([[1.0, -1.0]], [0.0]),
        ([[2.0]], [0.0]),
    ])
    x = [Value(0.5), Value(-0.25)]
    out = mlp(x)
    out[0].backward(setgrad=True)
    ok = (
        x[0].grad != 0.0 and
        x[1].grad != 0.0 and
        all(p.grad != 0.0 for p in mlp.parameters())
    )
    return check(
        "backward populates input and param grads",
        ok,
        f"x.grads={[xi.grad for xi in x]}, param.grads={[p.grad for p in mlp.parameters()]}"
    )

for fn in [test_backward_flows_to_inputs_and_params]:
    run_test(fn.__name__, fn)

print("\nDone. Any [FAIL] lines point to MLP logic that needs work.")

MLP — STRUCTURE
[PASS] MLP(3, [4, 4, 1]) creates 3 layers — len(layers)=3
[PASS] test_init_num_layers
[PASS] layer sizes match [2]->[3]->[1] — got [(2, 3), (3, 1)]
[PASS] test_layer_sizes
[PASS] all members are Layer objects
[PASS] test_layers_are_layer_objects

MLP — FORWARD PASS
[PASS] forward returns list[Value] of final layer size — type=list, len=1
[PASS] test_forward_output_shape
[PASS] forward matches stacked tanh layers — expected 0.000000, got 0.000000
[PASS] test_forward_expected_values
[PASS] forward works with scalar inputs — expected 0.416273, got 0.416273
[PASS] test_forward_scalar_inputs
[PASS] forward multi-output final layer — got [0.431808, 0.642015]
[PASS] test_forward_multi_output

MLP — PARAMETERS
[PASS] parameters returns all layer params — expected 13, got 13
[PASS] test_parameters_count
[PASS] parameters are Value objects
[PASS] test_parameters_are_values
[PASS] MLP(3, [4, 1]) has 21 parameters — got 21
[PASS] test_parameters_match_init_count

MLP — BACKWARD PAS

In [10]:
# --- End-to-end: train a tiny MLP (forward + backward + SGD) ---

random.seed(42)

# Binary classification-ish toy data (Karpathy lecture example)
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

# 3 -> 4 -> 4 -> 1 network
model = MLP(3, [4, 4, 1])

def predict(x):
    # Layer always returns a list; final layer has one neuron
    return model(x)[0]

def compute_loss():
    ypred = [predict(x) for x in xs]
    return sum((yout - ygt) ** 2 for yout, ygt in zip(ypred, ys))

print(f"params: {len(model.parameters())}")
print("initial preds:", [predict(x).data for x in xs])
loss0 = compute_loss()
print(f"initial loss: {loss0.data:.4f}")

lr = 0.1
steps = 100
losses = []

for k in range(steps):
    # forward
    loss = compute_loss()
    losses.append(loss.data)

    # zero grads, then backward
    for p in model.parameters():
        p.grad = 0.0
    loss.backward(setgrad=True)

    # SGD update
    for p in model.parameters():
        p.data += -lr * p.grad

    if k % 10 == 0 or k == steps - 1:
        print(f"step {k:3d}  loss={loss.data:.6f}")

final_preds = [predict(x).data for x in xs]
final_loss = compute_loss().data

print("\nfinal preds:", [f"{p:.4f}" for p in final_preds])
print("targets:    ", ys)
print(f"final loss:  {final_loss:.6f}")

# sanity checks: training actually exercised forward+backward
ok_decreased = final_loss < loss0.data
ok_signs = all((p > 0) == (t > 0) for p, t in zip(final_preds, ys))
print("\n[PASS]" if ok_decreased else "\n[FAIL]",
      f"loss decreased ({loss0.data:.4f} -> {final_loss:.4f})")
print("[PASS]" if ok_signs else "[FAIL]",
      "predictions match target signs")
print("[PASS]" if final_loss < 0.1 else "[FAIL]",
      f"loss below 0.1 (got {final_loss:.4f})")

params: 41
initial preds: [0.6994093620224069, 0.5026295816615511, 0.6931545900944501, 0.8755224728708614]
initial loss: 5.2305
step   0  loss=5.230518
step  10  loss=0.052920
step  20  loss=0.023549
step  30  loss=0.014791
step  40  loss=0.010671
step  50  loss=0.008298
step  60  loss=0.006763
step  70  loss=0.005692
step  80  loss=0.004904
step  90  loss=0.004301
step  99  loss=0.003869

final preds: ['0.9743', '-0.9701', '-0.9635', '0.9694']
targets:     [1.0, -1.0, -1.0, 1.0]
final loss:  0.003826

[PASS] loss decreased (5.2305 -> 0.0038)
[PASS] predictions match target signs
[PASS] loss below 0.1 (got 0.0038)
